# Final vOTU abundance and lifestyle summary
The shared filtered catalogue is retained in full. Abundance uses the existing coverage-filtered, normalized RPKM and count tables. Relative abundance is a percentage of catalogue RPKM, not of all sequencing reads. Microdiversity has a separate notebook.


## Notebook setup and plotting preferences


In [ ]:
# Load the libraries used by the workflow notebooks.
from pathlib import Path
import base64
import html
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO
from IPython.display import display

sns.set_style("ticks", {"axes.grid": True})
sns.set_palette("colorblind")
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Liberation Sans"],
                     "font.size": 9, "axes.linewidth": 0.5, "grid.linewidth": 0.25,
                     "svg.fonttype": "none", "savefig.dpi": 300, "figure.dpi": 150,
                     "text.usetex": False})
palette_lifestyle = {"Temperate": "#3399CC", "Virulent": "#FF9966",
                     "Unknown": "#AAAAAA", "Not applicable": "#CCCC99"}
palette_quality = {"Complete/High-quality": "#66CC99", "Medium-quality": "#666699",
                   "Low-quality/Not-determined": "#996699", "Unknown": "#AAAAAA"}


## Input files and shared variables


In [ ]:
input_fasta = snakemake.input.fasta
input_summary = snakemake.input.summary
SAMPLES = list(snakemake.params.samples)
group_column = snakemake.params.group_column
common_prevalence = float(snakemake.params.common_prevalence)
core_prevalence = float(snakemake.params.core_prevalence)
if len(SAMPLES) != len(set(SAMPLES)):
    raise ValueError("Sample identifiers must be unique")
if not 0 < common_prevalence <= core_prevalence <= 1:
    raise ValueError("Require 0 < common prevalence <= core prevalence <= 1")
if group_column and not snakemake.input.sample_metadata:
    raise ValueError("A grouping column requires abundance_sample_metadata")
for path in snakemake.output:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
figure_dir = Path(snakemake.output.figures)
figure_dir.mkdir(parents=True, exist_ok=True)
warnings_report = []
figure_files = []
desired_ranks = ["domain", "phylum", "class", "order", "family", "genus"]

def read_table(path, **kwargs):
    try:
        return pd.read_csv(path, **kwargs)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

def indexed(frame, column, source):
    if frame.empty and column not in frame:
        return pd.DataFrame(index=pd.Index([], name="vOTU"))
    if column not in frame:
        raise ValueError(f"{source}: missing identifier column {column}")
    if frame[column].isna().any() or frame[column].duplicated().any():
        raise ValueError(f"{source}: missing or duplicate identifiers")
    return frame.set_index(column).rename_axis("vOTU")

def save_figure(fig, name):
    fig.tight_layout()
    png = figure_dir / (name + ".png")
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(figure_dir / (name + ".svg"), bbox_inches="tight")
    figure_files.append(png)
    display(fig)
    plt.close(fig)


## Catalogue, mapping support and sample definitions
No control samples or vOTUs are excluded by name. Missing matrix rows/columns are errors, not invented zero measurements. All-zero libraries remain in the tables; their relative-abundance denominator is undefined.


In [ ]:
catalogue_rows = []
with open(input_fasta) as handle:
    for record in SeqIO.parse(handle, "fasta"):
        catalogue_rows.append([record.id, len(record.seq)])
catalogue = pd.DataFrame(catalogue_rows, columns=["vOTU", "sequence_length"])
catalogue = indexed(catalogue, "vOTU", "Final catalogue")
ids = catalogue.index

def read_matrix(path):
    frame = read_table(path, index_col=0)
    if frame.index.has_duplicates or frame.columns.has_duplicates:
        raise ValueError(f"Duplicate abundance identifiers in {path}")
    if not set(ids).issubset(frame.index) or not set(SAMPLES).issubset(frame.columns):
        raise ValueError(f"Catalogue IDs or samples missing from {path}")
    frame = frame.reindex(index=ids, columns=SAMPLES).apply(pd.to_numeric, errors="raise")
    if frame.isna().any().any() or not np.isfinite(frame.to_numpy(dtype=float)).all() or (frame < 0).any().any():
        raise ValueError(f"Missing, negative or non-finite measurements in {path}")
    return frame

filtered_RPKM_norm_df = read_matrix(snakemake.input.rpkm)
filtered_counts_norm_df = read_matrix(snakemake.input.counts)
coverage_df = read_matrix(snakemake.input.coverage)
if (coverage_df > 100).any().any():
    raise ValueError("Coverage percentages must not exceed 100")
presence_df = filtered_RPKM_norm_df.gt(0)
relative_norm_df = filtered_RPKM_norm_df.div(filtered_RPKM_norm_df.sum().replace(0, np.nan), axis=1) * 100
relative_norm_df.index.name = presence_df.index.name = "vOTU"
relative_abundance_columns = ["relative_abundance_" + sample for sample in SAMPLES]

metadata_samples = pd.DataFrame(index=pd.Index(SAMPLES, name="sample"))
if snakemake.input.sample_metadata:
    supplied = indexed(read_table(snakemake.input.sample_metadata[0], dtype={"sample": str}), "sample", "Sample metadata")
    metadata_samples = supplied.reindex(SAMPLES).rename_axis("sample")
    missing = sorted(set(SAMPLES) - set(supplied.index))
    if missing:
        warnings_report.append("Samples without metadata: " + ", ".join(missing))
    if group_column and group_column not in metadata_samples:
        raise ValueError(f"Grouping column {group_column} is missing")
display(metadata_samples)


## Viral identification, genome quality and sequence composition


In [ ]:
summary_df = read_table(input_summary)
summary_df = summary_df.loc[:, ~summary_df.columns.str.startswith("Unnamed:")]
summary_df = indexed(summary_df, "vOTU_name", "vOTU clustering summary").reindex(ids)
summary_df = summary_df.drop(columns=["sequence_length", "contig_name", "VirSorter2_seqname",
                                      "geNomad_seq_name", "geNomad_viral_name",
                                      "geNomad_viral_conservative_name"], errors="ignore")
metadata_df = catalogue.join(summary_df).rename(columns={"count_Y": "viral_detection_positive_tool_count"})
composition_columns = ["vOTU", "seqtk_seq_length", "seqtk_A_count", "seqtk_C_count", "seqtk_G_count",
                       "seqtk_T_count", "seqtk_2_count", "seqtk_3_count", "seqtk_4_count",
                       "seqtk_CpG_count", "seqtk_tv_count", "seqtk_ts_count", "seqtk_CpG_ts_count"]
composition = read_table(snakemake.input.composition, sep="\t", names=composition_columns)
composition = indexed(composition, "vOTU", "Sequence composition").reindex(ids)
for column in composition_columns[1:]:
    composition[column] = pd.to_numeric(composition[column], errors="raise")
composition["seqtk_GC"] = (composition["seqtk_G_count"] + composition["seqtk_C_count"]) / composition["seqtk_seq_length"].replace(0, np.nan)
metadata_df = metadata_df.join(composition)

bacphlip = read_table(snakemake.input.bacphlip, sep="\t", index_col=0)
if bacphlip.index.has_duplicates:
    raise ValueError("BACPHLIP identifiers must be unique")
for column in ["Virulent", "Temperate"]:
    if column not in bacphlip:
        bacphlip[column] = np.nan
bacphlip = bacphlip.reindex(ids)[["Virulent", "Temperate"]].apply(pd.to_numeric, errors="raise")
valid_scores = bacphlip.notna().all(axis=1) & bacphlip.ge(0).all(axis=1) & bacphlip.le(1).all(axis=1)
metadata_df["BACPHLIP_virulent_score"] = bacphlip["Virulent"]
metadata_df["BACPHLIP_temperate_score"] = bacphlip["Temperate"]
metadata_df["BACPHLIP_lifestyle"] = "Unknown"
metadata_df.loc[valid_scores, "BACPHLIP_lifestyle"] = np.where(bacphlip.loc[valid_scores, "Virulent"].ge(0.5), "Virulent", "Temperate")
metadata_df["BACPHLIP_lifestyle_source"] = np.where(valid_scores, "BACPHLIP_virulent_score_ge_0.5", "not_assessed_or_invalid_scores")
viral_group = metadata_df.get("VirSorter2_max_score_group", pd.Series("", index=ids)).fillna("")
rna_positive = metadata_df.get("VirSorter2_RNA_positive", pd.Series(False, index=ids)).astype(str).str.lower().eq("true")
not_phage_lifestyle = viral_group.isin(["RNA", "NCLDV", "lavidaviridae", "ssDNA"]) | rna_positive
metadata_df.loc[not_phage_lifestyle, "BACPHLIP_lifestyle"] = "Not applicable"
metadata_df.loc[not_phage_lifestyle, "BACPHLIP_lifestyle_source"] = "outside_dsDNA_phage_lifestyle_scope"
# VIBRANT lifecycle remains a separate evidence column; no inferred consensus.


## Viral taxonomy and reference similarity
Tool-specific assignments remain separate. Reference matches are similarities, not automatic species or host assignments. MetaVR metadata are reused only when that branch is enabled.


In [ ]:
phagcn = read_table(snakemake.input.phagcn)
if not phagcn.empty:
    phagcn = indexed(phagcn, "contig_name", "PhaGCN").rename(columns={"prediction": "PhaGCN_family", "score": "PhaGCN_score"})
    metadata_df = metadata_df.join(phagcn.reindex(ids)[["PhaGCN_family", "PhaGCN_score"]])
else:
    metadata_df["PhaGCN_family"] = pd.NA
    metadata_df["PhaGCN_score"] = np.nan
taxmyphage_path = Path(snakemake.input.taxmyphage) / "Summary_taxonomy.tsv"
if taxmyphage_path.is_file():
    taxmyphage = read_table(taxmyphage_path, sep="\t")
    taxmyphage = indexed(taxmyphage, "Genome", "TaxMyPhage").add_prefix("TaxMyPhage_")
    metadata_df = metadata_df.join(taxmyphage.reindex(ids))
else:
    warnings_report.append("No TaxMyPhage Summary_taxonomy.tsv was produced.")

blast_columns = ["qseqid", "sseqid", "salltitles", "qstart", "qend", "qlen", "slen", "qcovs", "evalue", "length", "pident"]
blast = read_table(snakemake.input.refseq, sep="\t", names=blast_columns, dtype=str, comment="#")
blast = blast.loc[blast["qseqid"].isin(ids)]
for column in blast_columns[3:]:
    blast[column] = pd.to_numeric(blast[column], errors="raise")
blast = blast.sort_values(["qseqid", "qcovs", "pident", "evalue", "length", "sseqid"],
                         ascending=[True, False, False, True, False, True], kind="mergesort").drop_duplicates("qseqid")
blast = indexed(blast, "qseqid", "RefSeq").rename(columns={
    "sseqid": "ViralRefSeq_subject_id", "salltitles": "ViralRefSeq_subject_title",
    "qstart": "ViralRefSeq_query_start", "qend": "ViralRefSeq_query_end",
    "qlen": "ViralRefSeq_query_length_bp", "slen": "ViralRefSeq_subject_length_bp",
    "qcovs": "ViralRefSeq_query_coverage_pct", "evalue": "ViralRefSeq_evalue",
    "length": "ViralRefSeq_hsp_length_bp", "pident": "ViralRefSeq_hsp_identity_pct"})
metadata_df = metadata_df.join(blast.reindex(ids))
if snakemake.input.metavr:
    metavr = indexed(read_table(snakemake.input.metavr[0], sep="\t"), "votu_id", "MetaVR summary")
    metavr_names = {"top_hit_sseqid": "METAVR_subject_id", "top_hit_qcovs": "METAVR_query_coverage_pct",
                   "top_hit_hsp_weighted_pident": "METAVR_hsp_weighted_identity_pct",
                   "top_hit_best_evalue": "METAVR_min_evalue"}
    metavr_columns = [c for c in metavr if c.startswith("top_hit_") or c.startswith("n_hits") or c == "n_metavr_hits"]
    metavr = metavr[metavr_columns].rename(columns={c: metavr_names.get(c, "METAVR_" + c.removeprefix("top_hit_")) for c in metavr_columns})
    metadata_df = metadata_df.join(metavr.reindex(ids))


## Host taxonomy and evidence
Keep the original iPHoP/GTDB lineage, confidence and selection reason. No implicit NCBI download, suffix stripping, isolate overrides or project-specific host exclusions are used. Optional informative CRISPR matches on microbial contigs longer than 5,000 bp take precedence, with disagreements recorded.


In [ ]:
iphop_path = Path(snakemake.input.iphop) / "Host_prediction_to_genus_m90.csv"
iphop = read_table(iphop_path) if iphop_path.is_file() else pd.DataFrame()
if not iphop_path.is_file():
    warnings_report.append("No iPHoP genus m90 table was produced; hosts remain unassigned.")
host_columns = ["Virus", "Host genus", "Confidence score", "List of methods", "AAI to closest RaFAH reference"]
if iphop.empty:
    iphop = pd.DataFrame(columns=host_columns)
elif not set(host_columns[:4]).issubset(iphop):
    raise ValueError("Unexpected iPHoP genus table columns")
iphop["Confidence score"] = pd.to_numeric(iphop["Confidence score"], errors="raise")
iphop = iphop.loc[iphop["Virus"].isin(ids) & iphop["Confidence score"].ge(90) & iphop["Host genus"].notna()].copy()
iphop["has_blast"] = iphop["List of methods"].str.contains("blast", case=False, na=False)
iphop = iphop.sort_values(["Virus", "has_blast", "Confidence score", "Host genus"],
                         ascending=[True, False, False, True], kind="mergesort").drop_duplicates("Virus")
iphop["iPHoP_selection_reason"] = np.where(iphop["has_blast"], "highest_confidence_blast", "highest_confidence_overall")
iphop = indexed(iphop.drop(columns="has_blast"), "Virus", "iPHoP").rename(columns={
    "Host genus": "iPHoP_host_lineage_raw", "Confidence score": "iPHoP_confidence_score",
    "List of methods": "iPHoP_list_of_methods", "AAI to closest RaFAH reference": "iPHoP_aai_to_closest_rafah_reference"})
iphop = iphop.reindex(ids)

def lineage_rank(value, prefix):
    if pd.isna(value):
        return pd.NA
    for part in str(value).split(";"):
        part = part.strip()
        if part.startswith(prefix + "__"):
            return part.split("__", 1)[1].strip() or pd.NA
    return pd.NA

for rank, prefix in zip(desired_ranks, ["d", "p", "c", "o", "f", "g"]):
    iphop["iPHoP_host_" + rank] = iphop["iPHoP_host_lineage_raw"].map(lambda value: lineage_rank(value, prefix))
iphop["iPHoP_host_genus_raw"] = iphop["iPHoP_host_genus"]
iphop["iPHoP_taxonomy_resolution_method"] = np.where(iphop["iPHoP_host_lineage_raw"].notna(), "recorded_GTDB_lineage_no_NCBI_translation", "unassigned")
metadata_df = metadata_df.join(iphop)

for rank in desired_ranks:
    metadata_df["CRISPR_host_" + rank] = pd.NA
metadata_df["CRISPR_match_count"] = 0
if snakemake.input.crispr:
    crispr_columns = ["spacer_acc", "phage_acc", "p_bh", "spacer_start", "spacer_end",
                      "phage_start", "phage_end", "putative_5_3_PAM", "npi"]
    crispr = read_table(snakemake.input.crispr[0], sep="\t", names=crispr_columns, comment="#")
    crispr = crispr.loc[crispr["phage_acc"].isin(ids)].copy()
    metadata_df["CRISPR_match_count"] = crispr["phage_acc"].value_counts().reindex(ids, fill_value=0)
    if snakemake.input.crispr_hosts:
        hosts = read_table(snakemake.input.crispr_hosts[0])
        required = {"microbial_contig", "microbial_contig_length_bp", "genus"}
        if not required.issubset(hosts):
            raise ValueError("CRISPR host metadata requires microbial_contig, microbial_contig_length_bp and genus")
        hosts = indexed(hosts, "microbial_contig", "CRISPR host metadata")
        hosts["microbial_contig_length_bp"] = pd.to_numeric(hosts["microbial_contig_length_bp"], errors="raise")
        crispr["microbial_contig"] = crispr["spacer_acc"].str.replace(r"^[#>]+", "", regex=True).str.extract(r"^(.+?)_CRISPR_", expand=False)
        crispr = crispr.join(hosts, on="microbial_contig", validate="many_to_one")
        for column in ["phage_start", "phage_end", "p_bh"]:
            crispr[column] = pd.to_numeric(crispr[column], errors="raise")
        crispr["alignment_length"] = (crispr["phage_end"] - crispr["phage_start"]).abs() + 1
        crispr = crispr.loc[crispr["microbial_contig_length_bp"].gt(5000) & crispr["genus"].notna() & crispr["genus"].str.strip().ne("")]
        crispr = crispr.sort_values(["phage_acc", "alignment_length", "p_bh", "microbial_contig_length_bp", "spacer_acc"],
                                   ascending=[True, False, True, False, True], kind="mergesort").drop_duplicates("phage_acc").set_index("phage_acc").reindex(ids)
        for rank in desired_ranks:
            if rank in crispr:
                metadata_df["CRISPR_host_" + rank] = crispr[rank]
        for source, target in {"spacer_acc": "spacer_accession", "microbial_contig": "microbial_contig",
                               "microbial_contig_length_bp": "microbial_contig_length_bp",
                               "p_bh": "p_bh", "alignment_length": "alignment_length_bp"}.items():
            metadata_df["CRISPR_" + target] = crispr[source]
    else:
        warnings_report.append("CRISPR matches counted without host assignment: no microbial host metadata supplied.")
elif snakemake.input.crispr_hosts:
    raise ValueError("CRISPR host metadata requires the microbial_spacers branch")

has_crispr = metadata_df["CRISPR_host_genus"].notna() & ~not_phage_lifestyle
has_iphop = metadata_df["iPHoP_host_genus"].notna() & ~not_phage_lifestyle
for rank in desired_ranks:
    metadata_df["resolved_host_" + rank] = metadata_df["CRISPR_host_" + rank].where(has_crispr).combine_first(metadata_df["iPHoP_host_" + rank].where(has_iphop & ~has_crispr))
metadata_df["host_assignment_source"] = np.select([has_crispr, has_iphop], ["CRISPR_SpacePHARER", "iPHoP_genus_m90"], default="unassigned")
metadata_df["host_assignment_rule"] = np.select([has_crispr & has_iphop, has_crispr, has_iphop],
    ["CRISPR_gt5000bp_override_iPHoP", "CRISPR_gt5000bp", "iPHoP_m90_prefer_BLAST_then_confidence"], default="unassigned")
metadata_df["CRISPR_iPHoP_genus_conflict"] = has_crispr & has_iphop & metadata_df["CRISPR_host_genus"].fillna("").ne(metadata_df["iPHoP_host_genus"].fillna(""))


## Metadata export — DANIEL column labels and ordering
Identification evidence, CheckV, sequence composition, BACPHLIP, viral taxonomy/reference similarity, resolved hosts and source-specific host evidence, then abundance. Original vOTU identifiers remain unchanged.


In [ ]:
metadata_df["mean_RPKM"] = filtered_RPKM_norm_df.mean(axis=1)
metadata_df["max_RPKM"] = filtered_RPKM_norm_df.max(axis=1)
# All-zero libraries have undefined relative abundance, not an invented percentage.
metadata_df["mean_relative_abundance"] = relative_norm_df.mean(axis=1)
metadata_df["max_relative_abundance"] = relative_norm_df.max(axis=1)
metadata_df = metadata_df.join(relative_norm_df.rename(columns=dict(zip(SAMPLES, relative_abundance_columns))))
tool_prefixes = {"checkV_": "CheckV_", "checkv_": "CheckV_", "bacphlip_": "BACPHLIP_",
                 "iphop_": "iPHoP_", "crispr_": "CRISPR_", "viral_refseq_": "ViralRefSeq_", "metavr_": "METAVR_"}
metadata_column_renames = {}
for column in metadata_df:
    for old_prefix, branded_prefix in tool_prefixes.items():
        if column.startswith(old_prefix):
            metadata_column_renames[column] = branded_prefix + column[len(old_prefix):]
            break
metadata_df = metadata_df.rename(columns=metadata_column_renames)
if metadata_df.columns.has_duplicates:
    raise ValueError("Metadata labels collide after tool-name capitalization")

def order_metadata_columns(dataframe):
    """Arrange metadata fields using the same topic ordering as DANIEL."""
    columns = list(dataframe.columns)
    ordered = []
    def add_exact(names):
        ordered.extend(name for name in names if name in columns and name not in ordered)
    def add_prefix(prefixes):
        ordered.extend(name for name in columns if name not in ordered and any(name.startswith(prefix) for prefix in prefixes))
    add_exact(["viral_detection_positive_tool_count"])
    add_prefix(["VirSorter2_"])
    add_prefix(["VIBRANT_"])
    add_exact([name for name in columns if name.startswith("geNomad_") and name != "geNomad_taxonomy"])
    add_prefix(["CheckV_"])
    add_exact(["seqtk_seq_length", "seqtk_GC", "seqtk_A_count", "seqtk_C_count", "seqtk_G_count", "seqtk_T_count",
               "seqtk_2_count", "seqtk_3_count", "seqtk_4_count", "seqtk_CpG_count", "seqtk_tv_count", "seqtk_ts_count", "seqtk_CpG_ts_count"])
    add_exact(["BACPHLIP_lifestyle", "BACPHLIP_lifestyle_source", "BACPHLIP_virulent_score", "BACPHLIP_temperate_score"])
    add_exact(["geNomad_taxonomy", "PhaGCN_family", "PhaGCN_score"])
    add_prefix(["ViralRefSeq_"])
    add_prefix(["METAVR_"])
    add_prefix(["resolved_host_"])
    add_exact(["host_assignment_source", "host_assignment_rule"])
    add_prefix(["iPHoP_"])
    add_prefix(["CRISPR_"])
    # Extra workflow fields have no DANIEL equivalent; keep them before abundance.
    abundance_columns = ["mean_RPKM", "max_RPKM", "mean_relative_abundance", "max_relative_abundance"] + relative_abundance_columns
    add_exact([name for name in columns if name not in abundance_columns])
    add_exact(abundance_columns)
    return dataframe.loc[:, ordered]

metadata_df = order_metadata_columns(metadata_df)
metadata_df.to_csv(snakemake.output.metadata)
relative_norm_df.to_csv(snakemake.output.relative)
presence_df.astype(int).to_csv(snakemake.output.presence)
display(metadata_df.head())


## Sample summaries and prevalence
Presence means nonzero abundance after the existing workflow filter. Common/core labels are reporting categories only, not additional catalogue filters. A sample with no detected vOTUs stays in the prevalence denominator.


In [ ]:
sample_summary = pd.DataFrame(index=pd.Index(SAMPLES, name="sample"))
sample_summary["detected_vOTUs"] = presence_df.sum()
sample_summary["filtered_normalised_counts"] = filtered_counts_norm_df.sum()
sample_summary["filtered_normalised_RPKM"] = filtered_RPKM_norm_df.sum()
sample_summary["mean_catalogue_breadth_percent"] = coverage_df.mean()
sample_summary["relative_abundance_defined"] = filtered_RPKM_norm_df.sum().gt(0)
sample_summary["status"] = np.where(sample_summary["relative_abundance_defined"], "Detected", "No detected vOTUs")
for column in metadata_samples:
    if column in sample_summary:
        raise ValueError(f"Sample metadata column collides with report metric: {column}")
sample_summary = sample_summary.join(metadata_samples)
sample_summary.to_csv(snakemake.output.samples)
prevalence = pd.DataFrame(index=ids)
prevalence["detected_samples"] = presence_df.sum(axis=1)
prevalence["total_samples"] = len(SAMPLES)
prevalence["prevalence"] = prevalence["detected_samples"] / len(SAMPLES) if SAMPLES else np.nan
prevalence["category"] = np.select(
    [prevalence["detected_samples"].eq(0), prevalence["prevalence"].eq(1),
     prevalence["prevalence"].ge(core_prevalence), prevalence["prevalence"].ge(common_prevalence)],
    ["Not detected", "Ubiquitous", "Core", "Common"], default="Rare")
prevalence = prevalence.join(metadata_df[["mean_relative_abundance", "BACPHLIP_lifestyle"]])
prevalence.to_csv(snakemake.output.prevalence)
display(sample_summary)
display(prevalence["category"].value_counts().rename("vOTUs").to_frame())


## Quality, lifestyle, taxonomy and host composition
Unknown and non-applicable categories remain in the denominator. Count proportions and abundance proportions answer different questions. High-quality views do not remove low/unknown-quality RNA candidates from the main catalogue.


In [ ]:
quality_raw = metadata_df.get("CheckV_quality", metadata_df.get("CheckV_checkv_quality", pd.Series(pd.NA, index=ids)))
quality = quality_raw.replace({"Complete": "Complete/High-quality", "High-quality": "Complete/High-quality",
                               "Medium-quality": "Medium-quality", "Low-quality": "Low-quality/Not-determined",
                               "Not-determined": "Low-quality/Not-determined"}).fillna("Unknown")
categories = {"Lifestyle": metadata_df["BACPHLIP_lifestyle"], "CheckV quality": quality,
              "Viral family": metadata_df["PhaGCN_family"].fillna("Unknown"),
              "Host genus": metadata_df["resolved_host_genus"].fillna("Unassigned")}
category_rows = []
for kind, labels in categories.items():
    for category in sorted(labels.unique()):
        selected = labels.eq(category)
        for sample in SAMPLES:
            detected = int(presence_df.loc[selected, sample].sum())
            total = int(presence_df[sample].sum())
            abundance = relative_norm_df.loc[selected, sample].sum(min_count=1)
            if sample_summary.loc[sample, "relative_abundance_defined"] and not selected.any():
                abundance = 0.0
            category_rows.append([kind, category, sample, int(selected.sum()), detected,
                                  detected * 100 / total if total else np.nan, abundance])
category_summary = pd.DataFrame(category_rows, columns=["kind", "category", "sample", "catalogue_vOTUs",
                                                       "detected_vOTUs", "detected_percent", "relative_abundance_percent"])
category_summary.to_csv(snakemake.output.categories, index=False)
genome_rows = []
for subset, selected in {"All vOTUs": pd.Series(True, index=ids),
                         "Complete/High-quality": quality.eq("Complete/High-quality")}.items():
    for feature in ["sequence_length", "seqtk_GC"]:
        values = pd.to_numeric(metadata_df.loc[selected, feature], errors="raise")
        weights = metadata_df.loc[selected, "mean_relative_abundance"]
        valid = values.notna() & weights.notna()
        denominator = weights[valid].sum()
        genome_rows.append([subset, feature, len(values), values.notna().sum(), values.mean(), values.median(),
                            values.min(), values.max(), (values[valid] * weights[valid]).sum() / denominator if denominator > 0 else np.nan])
genome_summary = pd.DataFrame(genome_rows, columns=["subset", "feature", "vOTUs", "measured_vOTUs",
                                                   "mean", "median", "min", "max", "abundance_weighted_mean"])
genome_summary.to_csv(snakemake.output.genomes, index=False)
display(genome_summary)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for row, kind in enumerate(["Lifestyle", "CheckV quality"]):
    frame = category_summary.loc[category_summary["kind"].eq(kind)]
    for col, metric in enumerate(["detected_percent", "relative_abundance_percent"]):
        matrix = frame.pivot(index="sample", columns="category", values=metric).reindex(SAMPLES)
        if matrix.shape[0] and matrix.shape[1] and matrix.notna().any().any():
            palette = palette_lifestyle if kind == "Lifestyle" else palette_quality
            matrix.plot.bar(stacked=True, ax=axes[row, col], color=[palette.get(c, "#AAAAAA") for c in matrix], width=0.8)
            axes[row, col].set_ylim(0, 100)
        else:
            axes[row, col].text(0.5, 0.5, "No detected vOTUs", ha="center", transform=axes[row, col].transAxes)
        axes[row, col].set_title(kind + ": " + metric.replace("_", " "))
        axes[row, col].set_ylabel("%")
save_figure(fig, "quality_lifestyle_composition")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, kind in zip(axes, ["Host genus", "Viral family"]):
    totals = relative_norm_df.mean(axis=1).groupby(categories[kind]).sum(min_count=1).sort_values(ascending=False).head(15).dropna()
    if len(totals):
        totals.sort_values().plot.barh(ax=ax)
    else:
        ax.text(0.5, 0.5, "No abundance data", ha="center", transform=ax.transAxes)
    ax.set_title(kind + " — top 15")
    ax.set_xlabel("Mean catalogue relative abundance (%)")
save_figure(fig, "host_taxonomy_abundance")


## Ranked abundance, genome features and prevalence


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
mean_relative = metadata_df["mean_relative_abundance"].dropna().sort_values(ascending=False)
if len(mean_relative):
    axes[0, 0].plot(np.arange(1, len(mean_relative) + 1), mean_relative.values)
    axes[0, 1].plot(np.arange(len(mean_relative) + 1), np.r_[0, mean_relative.cumsum()], label="All vOTUs")
    for name, color in palette_lifestyle.items():
        values = metadata_df.loc[metadata_df["BACPHLIP_lifestyle"].eq(name), "mean_relative_abundance"].dropna().sort_values(ascending=False)
        axes[0, 1].plot(np.arange(len(values) + 1), np.r_[0, values.cumsum()], label=name, color=color)
    axes[0, 1].legend()
axes[0, 0].set(xlabel="Abundance rank", ylabel="Mean relative abundance (%)")
axes[0, 1].set(xlabel="Abundance rank within category", ylabel="Cumulative relative abundance (%)", ylim=(0, 100))
for name, color in palette_lifestyle.items():
    selected = metadata_df["BACPHLIP_lifestyle"].eq(name)
    axes[1, 0].scatter(prevalence.loc[selected, "prevalence"], metadata_df.loc[selected, "mean_relative_abundance"], s=12, label=name, color=color)
axes[1, 0].set(xlabel="Prevalence (fraction of all samples)", ylabel="Mean relative abundance (%)", xlim=(0, 1))
axes[1, 1].scatter(metadata_df["sequence_length"] / 1000, metadata_df["seqtk_GC"] * 100, s=12)
axes[1, 1].set(xlabel="Sequence length (kb)", ylabel="GC (%)")
save_figure(fig, "abundance_genome_prevalence")

fig, ax = plt.subplots(figsize=(max(6, min(18, len(SAMPLES) * 0.5)), 9))
top_ids = metadata_df["mean_RPKM"].sort_values(ascending=False).head(30).index
if len(top_ids) and SAMPLES and relative_norm_df.loc[top_ids].notna().any().any():
    sns.heatmap(np.log10(relative_norm_df.loc[top_ids] + 1), ax=ax, cmap="viridis",
                cbar_kws={"label": "log10(relative abundance (%) + 1)"})
else:
    ax.text(0.5, 0.5, "No defined relative abundance", ha="center", transform=ax.transAxes)
ax.set_title("Top 30 vOTUs by mean filtered RPKM")
save_figure(fig, "top_vOTU_heatmap")


## Sample-accumulation curves
100 randomized sample orders, seed 42. Shading is the 2.5–97.5 percentile range across orders, not a confidence interval or sequencing-depth rarefaction.


In [ ]:
rng = np.random.default_rng(42)
accumulation_rows = []
if SAMPLES:
    curves = np.zeros((100, len(SAMPLES)), dtype=int)
    detected = presence_df.to_numpy(dtype=bool)
    for iteration in range(100):
        order = rng.permutation(len(SAMPLES))
        curves[iteration] = np.logical_or.accumulate(detected[:, order], axis=1).sum(axis=0)
    for step in range(len(SAMPLES)):
        accumulation_rows.append([step + 1, curves[:, step].mean(), *np.percentile(curves[:, step], [2.5, 97.5])])
accumulation = pd.DataFrame(accumulation_rows, columns=["samples", "mean_vOTUs", "lower_percentile", "upper_percentile"])
accumulation.to_csv(snakemake.output.accumulation, index=False)
fig, ax = plt.subplots(figsize=(6, 4))
if not accumulation.empty:
    ax.plot(accumulation["samples"], accumulation["mean_vOTUs"])
    ax.fill_between(accumulation["samples"].to_numpy(), accumulation["lower_percentile"].to_numpy(),
                    accumulation["upper_percentile"].to_numpy(), alpha=0.2)
ax.set(xlabel="Number of samples", ylabel="Cumulative detected vOTUs")
save_figure(fig, "sample_accumulation")


## Optional group summaries and shared ubiquitous vOTUs
Groups come only from the supplied sample metadata. No sample-name parsing, automatic control exclusions, or significance tests are performed. Ubiquitous means detected in every library of that group; missing group assignments are excluded from grouped panels only.


In [ ]:
group_rows = []
group_ubiquitous = pd.DataFrame(index=ids)
if group_column:
    group_values = metadata_samples[group_column].replace(r"^\s*$", np.nan, regex=True)
    if group_values.isna().any():
        warnings_report.append("Samples without a group are retained globally, but omitted from grouped panels.")
    for group in sorted(group_values.dropna().astype(str).unique()):
        members = group_values.index[group_values.astype(str).eq(group)].tolist()
        detections = presence_df[members]
        group_ubiquitous[group] = detections.all(axis=1)
        group_rows.append([group, len(members), ",".join(members), int(detections.any(axis=1).sum()),
                           int(detections.all(axis=1).sum()), detections.sum().mean()])
group_summary = pd.DataFrame(group_rows, columns=["group", "samples", "sample_ids", "detected_vOTUs",
                                                 "ubiquitous_vOTUs", "mean_detected_vOTUs_per_sample"])
group_summary.to_csv(snakemake.output.groups, index=False)
group_ubiquitous.astype(int).to_csv(snakemake.output.group_presence)
if len(group_ubiquitous.columns):
    # UpSet-style intersection bars and membership dots; retain all groups in the export.
    intersections = group_ubiquitous.loc[group_ubiquitous.any(axis=1)].apply(tuple, axis=1).value_counts().head(20)
    fig, (ax_bar, ax_dots) = plt.subplots(2, 1, figsize=(10, max(5, len(group_ubiquitous.columns) * 0.35 + 3)),
                                         gridspec_kw={"height_ratios": [2, 1]}, sharex=True)
    if len(intersections):
        ax_bar.bar(range(len(intersections)), intersections.values)
        for column, pattern in enumerate(intersections.index):
            active = np.flatnonzero(pattern)
            ax_dots.scatter([column] * len(pattern), range(len(pattern)), color="#DDDDDD", s=18)
            ax_dots.scatter([column] * len(active), active, color="#333333", s=22)
            if len(active) > 1:
                ax_dots.plot([column, column], [active.min(), active.max()], color="#333333", linewidth=1)
    ax_bar.set_ylabel("Ubiquitous vOTUs in intersection")
    ax_dots.set_yticks(range(len(group_ubiquitous.columns)), labels=group_ubiquitous.columns)
    ax_dots.set_xlabel("Top 20 nonempty group intersections")
    save_figure(fig, "group_ubiquitous_upset")
display(group_summary)


## Export the report and analysis settings


In [ ]:
settings = {"samples": SAMPLES, "catalogue": str(input_fasta),
            "presence": "filtered_RPKM_normalised > 0; existing coverage filters unchanged",
            "relative_abundance_denominator": "sum of filtered catalogue RPKM within each sample",
            "mean_relative_abundance": "mean over samples with a nonzero catalogue RPKM denominator",
            "common_prevalence": common_prevalence, "core_prevalence": core_prevalence,
            "group_column": group_column, "accumulation_permutations": 100, "random_seed": 42,
            "microdiversity": "separate notebook", "warnings": warnings_report}
sections = ["<html><head><meta charset='utf-8'><title>vOTU abundance and lifestyle</title></head><body>",
            "<h1>Final vOTU abundance and lifestyle</h1>",
            "<p>All final-catalogue vOTUs and samples are retained. Relative abundance is within the mapped catalogue, not all sequencing reads. "
            "Unknown predictions are retained; phage lifestyle and host assignments are not extended to RNA-virus candidates. "
            "The metadata CSV retains source-specific scores. No automatic statistical comparisons or sample exclusions are applied.</p>",
            "<h2>Settings and missing-data notes</h2><pre>" + html.escape(json.dumps(settings, indent=2)) + "</pre>",
            "<h2>Sample summary</h2>" + sample_summary.to_html(),
            "<h2>Genome summary</h2>" + genome_summary.to_html(index=False),
            "<h2>Group summary</h2>" + group_summary.to_html(index=False)]
for name, path in snakemake.output.items():
    if name not in ["html", "figures"]:
        relative_path = Path(path).relative_to(Path(snakemake.output.html).parent)
        sections.append('<p><a href="' + html.escape(str(relative_path), quote=True) + '">' + html.escape(name) + "</a></p>")
for path in figure_files:
    encoded = base64.b64encode(path.read_bytes()).decode()
    sections.append("<h2>" + html.escape(path.stem.replace("_", " ")) + "</h2><img style='max-width:100%' src='data:image/png;base64," + encoded + "'>")
sections.append("</body></html>")
Path(snakemake.output.html).write_text("\n".join(sections), encoding="utf-8")
print("Metadata:", snakemake.output.metadata)
print("Report:", snakemake.output.html)
